# Modelo de Evasão — Ensino Médio (Escola instituicao)

Prevê se o aluno **sai antes de terminar o ensino médio**, para a equipe de retenção
abordar a família antes da decisão.

| | |
|---|---|
| Base | `tabelas/extracao_medio.xlsx` |
| População | `MODALIDADE = 'EJND_ENS_MED'`, séries 1, 2 e 3 |
| Alvo | largou o ano depois do corte **ou** terminou e não voltou |
| Cortes | 30/04, 31/07 e 30/09 — três filas por ano |
| Origens | 2022 a 2025, validação em origem deslizante |

**Concluir o 3º ano é o caso negativo**, não exclusão — é assim que o modelo aprende
que o 3º ano tem risco baixo (1,0%) contra o 1º ano (20,3%).

O que cada corte enxerga muda, e é de propósito: em abril nenhum trimestre fechou,
em julho o T1 já saiu, em setembro o T1 e o T2.

## Tópico 1 — Carregamento dos Dados

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / 'tabelas').exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

XLSX = RAIZ / 'tabelas' / 'extracao_medio.xlsx'
CSV  = RAIZ / 'tabelas' / 'extracao_medio.csv'

# Le o mais recente dos dois. O XLSX fica travado quando esta aberto no Excel,
# e rodar sobre uma base velha sem perceber e pior do que falhar.
if XLSX.exists() and CSV.exists() and XLSX.stat().st_mtime < CSV.stat().st_mtime - 60:
    print('AVISO: o XLSX esta mais velho que o CSV -- usando o CSV.')
    print('       Feche o Excel e rode querys/extrair_medio.py para sincronizar.\n')
    origem = CSV
else:
    origem = XLSX if XLSX.exists() else CSV

if origem.suffix == '.xlsx':
    df = pd.read_excel(origem, dtype={'CURSO': str})
else:
    df = pd.read_csv(origem, encoding='utf-8-sig', dtype={'CURSO': str})

print(f'Base:    {origem.relative_to(RAIZ)}')
print(f'Formato: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Alunos distintos: {df.ALUNO.nunique()}  (cada um aparece ate 3x por ano)')
print()
print('Desfecho de cada linha:')
print(df.DESFECHO.value_counts().to_string())
df.head(3).drop(columns=['ALUNO'])   # ALUNO fora: RA de menor nao vai para o repositorio

## Tópico 2 — Filtro Obrigatório de Elegibilidade (`ELEGIVEL == 1`)

`ELEGIVEL = 0` marca **censura à direita**: o aluno largou o ano *antes* do corte,
então não há janela de observação nem o que prever.

Quem largou **depois** do corte fica — é exatamente o alvo.

In [ ]:
antes = len(df)
df = df[df['ELEGIVEL'] == 1].copy()
print(f'Antes: {antes} | Elegiveis: {len(df)} | Censurados: {antes - len(df)}')
print()

for chave, rotulo in [('TRIMESTRE', 'corte'), ('SERIE', 'serie'), ('TIPO_MEDIO', 'tipo')]:
    g = df.groupby(chave).agg(ALUNOS=('EVADIU', 'size'), EVASORES=('EVADIU', 'sum'))
    g['TAXA_%'] = (100 * g.EVASORES / g.ALUNOS).round(1)
    print(f'-- por {rotulo} --'); print(g.to_string()); print()

print(f'Taxa geral: {100 * df.EVADIU.mean():.1f}%')

## Tópico 3 — Engenharia de Features de Risco Combinado

Sinal isolado diz pouco; o que a coordenação reconhece é o **acúmulo**. Aqui as
features de risco usam nota **por trimestre**, que é o que existe na data do corte.

In [ ]:
d = df

# --- risco academico do trimestre fechado ---------------------------------
d['RISCO_NOTA_T1']    = (d['NOTA_T1'] < 6.0).astype('float')     # NaN se T1 nao fechou
d['RISCO_NOTA_T2']    = (d['NOTA_T2'] < 6.0).astype('float')
d['RISCO_MUITA_REC']  = (d['RECUPERACOES_T1'].fillna(0) >= 4).astype(int)
d['QUEDA_T1_T2']      = (d['DELTA_T1_T2'] < -1.0).astype('float')
d['QUEDA_VS_ANO_ANT'] = (d['DELTA_ANO_ANTERIOR'] < -1.0).astype('float')

# --- risco de frequencia ---------------------------------------------------
d['RISCO_FALTAS'] = (d['FALTAS_POR_MES'] > d['FALTAS_POR_MES'].median() * 2).astype('float')

# --- risco financeiro ------------------------------------------------------
d['PERC_EM_ABERTO'] = d['FIN_QTD_EM_ABERTO'] / d['FIN_QTD_TITULOS'].replace(0, np.nan)
d['RISCO_INADIMPLENCIA'] = (d['PERC_EM_ABERTO'].fillna(0) > 0.20).astype(int)

# --- risco de vinculo ------------------------------------------------------
d['RISCO_VINCULO_FRACO'] = ((d['NOVO_NA_ESCOLA'] == 1) | (d['ANOS_DE_CASA'] <= 1)).astype(int)

# --- desengajamento do portal ----------------------------------------------
d['RISCO_SUMIDO_30D'] = (d['PORTAL_DIAS_SEM_ACESSAR'].fillna(999) > 30).astype(int)

# --- indice combinado ------------------------------------------------------
d['RISCO_COMBINADO'] = (d[['RISCO_NOTA_T1', 'RISCO_FALTAS', 'RISCO_INADIMPLENCIA',
                           'RISCO_VINCULO_FRACO', 'RISCO_SUMIDO_30D']]
                        .fillna(0).sum(axis=1))

novas = ['RISCO_NOTA_T1','RISCO_NOTA_T2','RISCO_MUITA_REC','QUEDA_T1_T2',
         'QUEDA_VS_ANO_ANT','RISCO_FALTAS','PERC_EM_ABERTO','RISCO_INADIMPLENCIA',
         'RISCO_VINCULO_FRACO','RISCO_SUMIDO_30D','RISCO_COMBINADO']
print(f'{len(novas)} features criadas\n')

g = d.groupby('RISCO_COMBINADO').agg(ALUNOS=('EVADIU','size'), EVASORES=('EVADIU','sum'))
g['TAXA_%'] = (100 * g.EVASORES / g.ALUNOS).round(1)
print('Evasao por nivel de RISCO_COMBINADO:')
print(g.to_string())

## Tópico 4 — Separação de `XXX` (Features) e `yyy` (Target)

| Coluna | Por quê sai |
|---|---|
| `DESFECHO` | Diz literalmente o desfecho — é o rótulo em texto |
| `DT_SAIDA` | Data do próprio evento |
| `ALUNO` | Identificador — o modelo decora o aluno |
| `COORTE`, `DT_CORTE` | Auditoria e chave temporal |
| `ELEGIVEL` | Filtro de controle, já aplicado |
| `NOME_CURSO` | Redundante com `CURSO` |
| `SEXO` | Ressalva ética — só auditoria de viés, nunca no `X` |

`TRIMESTRE` **fica** como feature: a taxa base muda entre os cortes (13,4% / 11,1% /
9,7%) e o modelo precisa dessa referência.

In [ ]:
COLUNAS_REMOVER = [
    'DESFECHO', 'DT_SAIDA',          # vazamento direto
    'ALUNO',                          # identificador
    'COORTE', 'DT_CORTE',             # auditoria / chave temporal
    'ELEGIVEL',                       # filtro ja aplicado
    'NOME_CURSO',                     # redundante com CURSO
    'SEXO',                           # so auditoria de vies
    'EVADIU',                         # alvo
]

XXX = d.drop(columns=[c for c in COLUNAS_REMOVER if c in d.columns])
yyy = d['EVADIU'].astype(int)

ids_aluno  = d['ALUNO'].copy()
coortes    = d['COORTE'].copy()
trimestres = d['TRIMESTRE'].copy()
sexo_audit = d['SEXO'].copy()

print(f'XXX: {XXX.shape[0]} linhas x {XXX.shape[1]} features')
print(f'yyy: {yyy.sum()} evasores de {len(yyy)}  ({100*yyy.mean():.1f}%)')

## Tópico 5 — Encoding de Categóricas e Imputação pela Mediana

Atenção ao que o nulo significa aqui. Em `NOTA_T1` no corte de abril, nulo quer dizer
**"o trimestre ainda não fechou"**, não "aluno sem nota". A informação está preservada
em `TRIMESTRE`, que diz qual corte é — então a mediana preenche o valor sem apagar o
sinal.

`CURSO` é **código, não número**: entra como categoria.

In [ ]:
from sklearn.impute import SimpleImputer

cat = XXX.select_dtypes(include=['object', 'string']).columns.tolist()
print(f'Categoricas ({len(cat)}): {cat}')

X2 = XXX.copy()
for c in cat:
    freq = X2[c].value_counts()
    raros = freq[freq < 50].index
    if len(raros):
        X2[c] = X2[c].where(~X2[c].isin(raros), 'OUTRO')
        print(f'  {c}: {len(raros)} niveis raros -> OUTRO ({X2[c].nunique()} niveis)')

X_encoded = pd.get_dummies(X2, columns=cat)

# colunas 100% nulas nao sobrevivem a imputacao pela mediana
vazias = X_encoded.columns[X_encoded.notna().sum() == 0].tolist()
if vazias:
    print(f'\nColunas 100% nulas removidas: {vazias}')
    X_encoded = X_encoded.drop(columns=vazias)

nulos = X_encoded.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(f'\nColunas com nulo ({len(nulos)}):')
print(nulos.to_string())

imputer = SimpleImputer(strategy='median')
X_encoded = pd.DataFrame(imputer.fit_transform(X_encoded),
                         columns=X_encoded.columns, index=X_encoded.index)
print(f'\nApos encoding e imputacao: {X_encoded.shape[1]} colunas, '
      f'{int(X_encoded.isna().sum().sum())} nulos')

## Tópico 6 — Divisão Treino (80%) e Teste (20%) Estratificado

Split estratificado, conforme o processo padrão — **agrupado por aluno**.

> O mesmo aluno aparece até 9 vezes na base (3 cortes × 3 anos). Um split puramente
> aleatório colocaria a mesma pessoa nos dois lados e o modelo decoraria o aluno em vez
> de aprender o padrão. `GroupShuffleSplit` mantém cada aluno inteiro de um lado só.
>
> Ainda assim, **este número é otimista**: ele embaralha as coortes, então o modelo
> enxerga 2025 para prever 2023. O número honesto está no anexo.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_tr, idx_te = next(gss.split(X_encoded, yyy, groups=ids_aluno))

X_train, X_test = X_encoded.iloc[idx_tr], X_encoded.iloc[idx_te]
y_train, y_test = yyy.iloc[idx_tr], yyy.iloc[idx_te]

print(f'Treino: {len(X_train)} linhas | {y_train.sum()} evasores ({100*y_train.mean():.1f}%)')
print(f'Teste : {len(X_test)} linhas | {y_test.sum()} evasores ({100*y_test.mean():.1f}%)')

vaz = set(ids_aluno.iloc[idx_tr]) & set(ids_aluno.iloc[idx_te])
print(f'\nAlunos nos dois lados: {len(vaz)}  (tem que ser 0)')

## Tópico 7 — Treinamento do Random Forest (`class_weight='balanced'`)

Base desbalanceada (~11% de evasão). Sem `class_weight` o modelo chuta "ninguém evade"
e acerta 89%.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
modelo.fit(X_train, y_train)
y_proba = modelo.predict_proba(X_test)[:, 1]

print(f'Treinado: {modelo.n_estimators} arvores, {X_train.shape[1]} features')
print(f'Probabilidade -- min {y_proba.min():.3f} | media {y_proba.mean():.3f} | max {y_proba.max():.3f}')

## Tópico 8 — Otimização do Limiar de Decisão (0,20 a 0,50)

O limiar de 0,50 é arbitrário e, numa base desbalanceada, deixa evasor passar. Abaixar
aumenta o recall ao custo de uma fila maior.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

linhas = []
for t in np.arange(0.20, 0.51, 0.01):
    yp = (y_proba >= t).astype(int)
    linhas.append({'LIMIAR': round(t, 2),
                   'PRECISAO': round(precision_score(y_test, yp, zero_division=0), 3),
                   'RECALL': round(recall_score(y_test, yp, zero_division=0), 3),
                   'F1': round(f1_score(y_test, yp, zero_division=0), 3),
                   'FILA': int(yp.sum()),
                   'FILA_%': round(100 * yp.mean(), 1)})
tuning = pd.DataFrame(linhas)
print(tuning.to_string(index=False))
melhor = tuning.loc[tuning.F1.idxmax()]
print(f"\nMelhor F1 no limiar {melhor.LIMIAR}: F1={melhor.F1} | recall={melhor.RECALL}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tuning.LIMIAR, tuning.PRECISAO, marker='o', ms=3, label='Precisão')
ax.plot(tuning.LIMIAR, tuning.RECALL,   marker='s', ms=3, label='Recall')
ax.plot(tuning.LIMIAR, tuning.F1,       marker='^', ms=3, label='F1')
ax.axvline(0.35, color='#C53030', ls='--', lw=1.5, label='Adotado (0,35)')
ax.axvline(0.50, color='#718096', ls=':',  lw=1.5, label='Padrão (0,50)')
ax.set_xlabel('Limiar'); ax.set_ylabel('Métrica')
ax.set_title('Threshold Tuning — Ensino Médio')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Tópico 9 — Avaliação com Limiar Otimizado (`THRESHOLD = 0.30`)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

THRESHOLD = 0.30
y_pred_035 = (y_proba >= THRESHOLD).astype(int)
y_pred_050 = (y_proba >= 0.50).astype(int)

print(f'=== Limiar {THRESHOLD} ===')
print(classification_report(y_test, y_pred_035, target_names=['Permanece', 'Evade'], digits=3))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.3f}')
print(f'PR-AUC  : {average_precision_score(y_test, y_proba):.3f}')
print()

# recall@K -- traduz a capacidade real da equipe de retencao
ordem = np.argsort(-y_proba)
yt = np.asarray(y_test)[ordem]
print('Fila ordenada por risco:')
for pct in (0.05, 0.10, 0.15, 0.20, 0.30):
    k = max(1, int(len(yt) * pct))
    print(f'  topo {pct:>4.0%} ({k:4d} linhas) -> recall {yt[:k].sum()/yt.sum():6.1%} '
          f'| precisao {yt[:k].mean():6.1%}')

## Tópico 10 — Matrizes de Confusão Comparativas (0,50 vs 0,35)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yp, titulo in [(axes[0], y_pred_050, 'Limiar padrão — 0,50'),
                       (axes[1], y_pred_035, 'Limiar otimizado — 0,35')]:
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Permanece', 'Evade'], yticklabels=['Permanece', 'Evade'],
                annot_kws={'size': 14})
    vn, fp, fn, vp = cm.ravel()
    ax.set_title(f'{titulo}\nCapturados: {vp} de {vp+fn}  |  Falsos alarmes: {fp}')
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

for nome, yp in [('0,50', y_pred_050), ('0,35', y_pred_035)]:
    vn, fp, fn, vp = confusion_matrix(y_test, yp).ravel()
    print(f'Limiar {nome}: recall {vp/(vp+fn):.1%} | precisao {vp/(vp+fp) if vp+fp else 0:.1%} '
          f'| fila de {vp+fp}')

## Tópico 11 — Curvas ROC e Precision-Recall

Com ~11% de evasão, a curva **Precision-Recall** é a que informa. A ROC parece boa
mesmo quando o modelo não presta, porque a classe negativa domina.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(y_test, y_proba)
prec, rec, _ = precision_recall_curve(y_test, y_proba)
auc_roc = roc_auc_score(y_test, y_proba)
auc_pr  = average_precision_score(y_test, y_proba)
base = y_test.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color='#2B6CB0', lw=2, label=f'ROC-AUC = {auc_roc:.3f}')
axes[0].plot([0, 1], [0, 1], ls='--', color='#A0AEC0', label='Aleatório')
axes[0].set_xlabel('Falso positivo'); axes[0].set_ylabel('Verdadeiro positivo')
axes[0].set_title('Curva ROC'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(rec, prec, color='#1A365D', lw=2, label=f'PR-AUC = {auc_pr:.3f}')
axes[1].axhline(base, ls='--', color='#A0AEC0', label=f'Base ({base:.1%})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precisão')
axes[1].set_title('Curva Precision-Recall'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'ROC-AUC {auc_roc:.3f} | PR-AUC {auc_pr:.3f} | ganho sobre a base: {auc_pr/base:.1f}x')

## Tópico 12 — Importância das Features (Top 15)

Aqui se vigia vazamento residual. Todo o bloco acadêmico vem de trimestre **já fechado
na data do corte**, e a recuperação do T2 foi deixada de fora de propósito (em setembro
só 48,9% estava lançada — viria completa no treino e pela metade em produção).

In [ ]:
imp = (pd.DataFrame({'FEATURE': X_encoded.columns, 'IMPORTANCIA': modelo.feature_importances_})
       .sort_values('IMPORTANCIA', ascending=False).reset_index(drop=True))

top15 = imp.head(15)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15.FEATURE[::-1], top15.IMPORTANCIA[::-1], color='#2B6CB0')
ax.set_xlabel('Importância'); ax.set_title('Top 15 Features — Ensino Médio')
ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()

print(top15.to_string(index=False))
print(f'\nAs 15 primeiras concentram {100*top15.IMPORTANCIA.sum():.1f}% da importancia')
sem_peso = imp[imp.IMPORTANCIA < 0.002]
print(f'\nImportancia < 0,002 (candidatas a corte): {len(sem_peso)}')
print(sem_peso.FEATURE.tolist())

## Tópico 13 — Salvamento do Artefato em Pickle

In [ ]:
import joblib, sklearn

DEST_ML = RAIZ / 'ml' / 'escola-basica'
DEST_ML.mkdir(parents=True, exist_ok=True)
PKL = DEST_ML / 'modelo_evasao_medio.pkl'

artefato = {
    'modelo': modelo,
    'imputer': imputer,
    'colunas': list(X_encoded.columns),
    'colunas_categoricas': cat,
    'colunas_removidas': COLUNAS_REMOVER,
    'threshold': THRESHOLD,
    'metricas_holdout': {
        'roc_auc': float(auc_roc), 'pr_auc': float(auc_pr),
        'recall@0.35': float(recall_score(y_test, y_pred_035)),
        'precisao@0.35': float(precision_score(y_test, y_pred_035, zero_division=0)),
    },
    'base': 'tabelas/extracao_medio.xlsx',
    'alvo': 'saiu antes de terminar o ensino medio',
    'cortes': {1: '30/04', 2: '31/07', 3: '30/09'},
    'coortes': sorted(coortes.unique().tolist()),
    'versao_sklearn': sklearn.__version__,
}
joblib.dump(artefato, PKL)
print(f'Salvo: {PKL.relative_to(RAIZ)}  ({PKL.stat().st_size/1024:.0f} KB)')
print(f'Features esperadas na inferencia: {len(artefato["colunas"])}')

## Tópico 14 — Matriz de Correlação de Pearson

Três entregas: heatmap triangular ordenado, ranking das features mais correlacionadas
com `EVADIU` e diagnóstico de multicolinearidade (`|r| ≥ 0,70`).

In [ ]:
num_corr = X_encoded.select_dtypes(include=[np.number]).copy()
num_corr = num_corr.loc[:, num_corr.std() > 0]
num_corr['EVADIU'] = yyy.values

corr_alvo = num_corr.corr(numeric_only=True)['EVADIU'].drop('EVADIU').sort_values(ascending=False)
ordem = ['EVADIU'] + corr_alvo.abs().sort_values(ascending=False).head(24).index.tolist()
M = num_corr[ordem].corr(numeric_only=True)

DEST_PNG = RAIZ / 'relatorios' / 'matriz-correlacao'
DEST_PNG.mkdir(parents=True, exist_ok=True)
ARQ_PNG = DEST_PNG / 'matriz_correlacao_medio.png'

mask = np.triu(np.ones_like(M, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(11, 9), dpi=150)
sns.heatmap(M, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={'shrink': .7, 'label': 'Pearson r'},
            annot=False, ax=ax)
ax.set_title('Matriz de Correlação de Pearson — Ensino Médio\n(EVADIU na 1ª coluna)',
             fontsize=13, pad=14)
plt.tight_layout()
fig.set_size_inches(11, 9)
fig.savefig(ARQ_PNG, dpi=150)          # 1650 x 1350 px
plt.show()

from PIL import Image
print(f'Salvo: {ARQ_PNG.relative_to(RAIZ)}  {Image.open(ARQ_PNG).size} px')

### 14.b — Ranking de correlação com `EVADIU`

`+r` empurra para a evasão (risco), `−r` segura na escola (retenção).

In [ ]:
top_pos, top_neg = corr_alvo.head(12), corr_alvo.tail(12)
rank = pd.concat([top_pos, top_neg]).sort_values()

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(rank.index, rank.values, color=['#C53030' if v > 0 else '#2B6CB0' for v in rank.values])
ax.axvline(0, color='#2D3748', lw=1)
ax.set_xlabel('Correlação de Pearson com EVADIU')
ax.set_title('Risco (+r, vermelho)  ×  Retenção (−r, azul)')
ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()

print('--- Maior risco ---');      print(top_pos.round(3).to_string())
print('\n--- Maior retencao ---'); print(top_neg.round(3).to_string())

### 14.c — Diagnóstico de multicolinearidade (`|r| ≥ 0,70`)

Serve para **interpretar**, não para podar: testes de ablação nesta base mostraram que
cortar pares redundantes **piora** o Random Forest, que sorteia features por árvore.

In [ ]:
c = num_corr.drop(columns=['EVADIU']).corr(numeric_only=True).abs()
tri = c.where(np.triu(np.ones(c.shape), k=1).astype(bool)).stack()
multi = (tri[tri >= 0.70].reset_index()
         .rename(columns={'level_0': 'FEATURE_A', 'level_1': 'FEATURE_B', 0: 'R_ABS'})
         .sort_values('R_ABS', ascending=False).reset_index(drop=True))
multi['R_ABS'] = multi.R_ABS.round(3)
print(f'{len(multi)} pares com |r| >= 0,70:' if len(multi) else 'Nenhum par com |r| >= 0,70.')
if len(multi):
    print(multi.to_string(index=False))

---

# Anexo A — Validação em Origem Deslizante

O Tópico 6 embaralha as coortes. Aqui o modelo treina só no **passado** e é testado no
**ano seguinte**, que é como ele será usado.

A base tem 4 anos observados, o que rende **3 dobras** — sempre é preciso um ano para
treinar e o próximo para testar.

> Olhe a **dobra 3**, que testa na coorte 2025. É ela que estima a próxima turma.

In [ ]:
X_orig = X_encoded.copy()
X_orig['__coorte'] = coortes.values

resultados = []
for origem in sorted(coortes.unique())[:-1]:
    tr = X_orig['__coorte'] <= origem
    te = X_orig['__coorte'] == origem + 1
    if te.sum() == 0:
        continue
    Xtr, Xte = X_orig.loc[tr].drop(columns='__coorte'), X_orig.loc[te].drop(columns='__coorte')
    ytr, yte = yyy[tr.values], yyy[te.values]

    m = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    yy = np.asarray(yte)[np.argsort(-p)]
    k10 = max(1, int(len(yy) * 0.10))

    resultados.append({
        'TREINA_ATE': int(origem), 'TESTA_EM': int(origem) + 1,
        'N_TESTE': int(len(yte)), 'EVASORES': int(yte.sum()),
        'BASE_%': round(100 * yte.mean(), 1),
        'PR_AUC': round(average_precision_score(yte, p), 3),
        'ROC_AUC': round(roc_auc_score(yte, p), 3),
        'RECALL@10%': round(yy[:k10].sum() / yy.sum(), 3),
        'RECALL@20%': round(yy[:2*k10].sum() / yy.sum(), 3),
    })

origens = pd.DataFrame(resultados)
print(origens.to_string(index=False))
ult = origens.iloc[-1]
print(f"\nULTIMA ORIGEM (treina ate {int(ult.TREINA_ATE)}, testa em {int(ult.TESTA_EM)}):")
print(f"  PR-AUC {ult.PR_AUC} | recall@10% {ult['RECALL@10%']:.1%} | recall@20% {ult['RECALL@20%']:.1%}")
print(f"\nHoldout do Topico 9 -> PR-AUC {auc_pr:.3f}")
print(f"Ultima origem       -> PR-AUC {ult.PR_AUC}")

# Anexo B — Desempenho por corte trimestral

Cada corte enxerga uma coisa diferente. Abril não tem nota nenhuma; julho tem o T1;
setembro tem o T1 e o T2. É esperado que o poder preditivo cresça ao longo do ano.

In [ ]:
CORTE_NOME = {1: '30/04  (sem nota)', 2: '31/07  (T1)', 3: '30/09  (T1+T2)'}
linhas = []
for tri in sorted(trimestres.unique()):
    sel = (trimestres == tri).values
    Xs, ys, cs = X_encoded[sel], yyy[sel], coortes[sel]
    tr, te = (cs <= 2024).values, (cs == 2025).values
    if te.sum() == 0:
        continue
    m = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
    m.fit(Xs[tr], ys[tr])
    p = m.predict_proba(Xs[te])[:, 1]
    yy = np.asarray(ys[te])[np.argsort(-p)]
    k = max(1, int(len(yy) * 0.10))
    linhas.append({'CORTE': CORTE_NOME[int(tri)], 'N': int(te.sum()),
                   'EVASORES': int(ys[te].sum()), 'BASE_%': round(100*ys[te].mean(), 1),
                   'RECALL@10%': round(yy[:k].sum()/yy.sum(), 3),
                   'RECALL@20%': round(yy[:2*k].sum()/yy.sum(), 3),
                   'PR_AUC': round(average_precision_score(ys[te], p), 3)})
print('Treina em <=2024, testa na coorte 2025:')
print(pd.DataFrame(linhas).to_string(index=False))

# Anexo C — Auditoria de viés (`SEXO`)

`SEXO` ficou **fora** do `X`. Aqui só se verifica se o modelo alerta os grupos de forma
equilibrada — que é o uso legítimo da variável.

In [ ]:
aud = pd.DataFrame({
    'SEXO': sexo_audit.iloc[idx_te].values,
    'REAL': np.asarray(y_test),
    'ALERTA': y_pred_035,
    'PROBA': y_proba,
})
g = aud.groupby('SEXO').agg(LINHAS=('REAL','size'), EVASAO_REAL=('REAL','mean'),
                            TAXA_ALERTA=('ALERTA','mean'), PROBA_MEDIA=('PROBA','mean')).round(3)
print(g.to_string())
print('\nRecall por grupo (dos que evadiram, quantos foram alertados):')
print(aud[aud.REAL == 1].groupby('SEXO').ALERTA.mean().round(3).to_string())